<a href="https://colab.research.google.com/github/BeeMugo9/NewsAPI-Classifier/blob/main/Code_assessment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### What this notebook does

This notebook retrieves news articles from the public NewsAPI about humanitarian and medical contexts in countries where MSF operates. For each article it:

1. Retrieves articles using the NewsAPI free tier
2. Extracts key fields: title, source, date, description, URL
3. Classifies each article into one of five categories: Health and Disease, Conflict and Security, Natural Disaster, Humanitarian Response, or General
4. Adds sentiment: Positive, Neutral, or Negative based on the article description
5. Outputs the results as both CSV and JSON

### Classification approach

Classification uses a keyword scoring method. Each article's title and description are scanned for words associated with each category. The category with the highest keyword match count wins. This approach is transparent, auditable, and requires no model training — any MSF analyst can read the keyword lists and understand exactly how every classification decision was made.

### MSF countries covered by queries
South Sudan, DRC, Yemen, Syria, Afghanistan, Somalia, Nigeria, Ethiopia, Central African Republic, Niger

###import libraries

In [1]:
import pandas as pd
import numpy as np

import subprocess
subprocess.run(['pip', 'install', 'requests', 'pandas', 'textblob', '--quiet'], check=True)

import requests
import json
from textblob import TextBlob
from datetime import datetime, timedelta

print("All libraries loaded successfully.")
print(f"Run date: {datetime.now().strftime('%d %B %Y')}")

All libraries loaded successfully.
Run date: 27 April 2026


###Configuration

Created a free account on https://newsapi.org and copied the API key and pasted below

In [2]:
MYAPIKEY  = "5a1ccxxxxxxxxxxxxxxxxxxxxxxx10fa2ac"
MYBASEURL = "https://newsapi.org/v2/everything"

In [3]:
# Search queries targeting humanitarian and medical contexts
QUERIES = [
    "humanitarian crisis MSF South Sudan",
    "medical aid conflict Yemen Syria",
    "disease outbreak DRC Congo Africa",
    "refugee health Somalia Ethiopia",
    "cholera malaria Nigeria Niger",
    "emergency relief Afghanistan",
    "MSF Medecins Sans Frontieres health",
    "famine food crisis Central African Republic",
]

In [4]:
# Retrieve articles for the past 30 days
FROM_DATE = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')

print(f"Configuration ready.")
print(f"Queries to run: {len(QUERIES)}")
print(f"Date range: {FROM_DATE} to today")

Configuration ready.
Queries to run: 8
Date range: 2026-03-28 to today


###Retrieve the articles from the newsAPI

In [5]:
def fetch_articles(query, from_date, api_key, page_size=20):
    """
    Retrieve articles from NewsAPI for.
    Returns a list of article dictionaries.
    """
    params = {
        "q":        query,
        "from":     from_date,
        "language": "en",
        "sortBy":   "relevancy",
        "pageSize": page_size,
        "apiKey":   api_key
    }

    response = requests.get(MYBASEURL, params=params, timeout=10)

    if response.status_code == 200:
        data = response.json()
        return data.get("articles", [])
    else:
        print(f"  Warning: Query '{query[:40]}' returned status {response.status_code}")
        return []


print("Retrieving articles from NewsAPI...")
print("-" * 55)

all_raw_articles = []

for query in QUERIES:
    articles = fetch_articles(query, FROM_DATE, MYAPIKEY)
    all_raw_articles.extend(articles)
    print(f"  '{query[:45]}': {len(articles)} articles")

print("-" * 45)
print(f"Total articles before deduplication: {len(all_raw_articles)}")

Retrieving articles from NewsAPI...
-------------------------------------------------------
  'humanitarian crisis MSF South Sudan': 7 articles
  'medical aid conflict Yemen Syria': 3 articles
  'disease outbreak DRC Congo Africa': 3 articles
  'refugee health Somalia Ethiopia': 0 articles
  'cholera malaria Nigeria Niger': 2 articles
  'emergency relief Afghanistan': 17 articles
  'MSF Medecins Sans Frontieres health': 1 articles
  'famine food crisis Central African Republic': 2 articles
---------------------------------------------
Total articles before deduplication: 35


###Clean, extract the key fields and check duplication

In [6]:
def extract_fields(article):
    """
    Extract the required fields from every raw NewsAPI article object.
    Returns a clean dictionary with five fields.
    """
    return {
        "title":       (article.get("title")       or "").strip(),
        "source":      (article.get("source", {}).get("name") or "").strip(),
        "published_date": (article.get("publishedAt") or "")[:10],  # date only
        "description": (article.get("description")  or "").strip(),
        "url":         (article.get("url")           or "").strip(),
    }

In [7]:
# Extract the fields from all articles
records = [extract_fields(a) for a in all_raw_articles]


In [8]:
# Load into a DataFrame for easy manipulation
df = pd.DataFrame(records)

In [9]:
# Remove duplicates by title, which are basically same story from multiple queries
df = df.drop_duplicates(subset=["title"])


In [10]:
# Remove rows where title is empty or the NewsAPI placeholder is [Removed]
df = df[df["title"].str.len() > 0]
df = df[df["title"] != "[Removed]"]

In [11]:
# Reset index after cleaning
df = df.reset_index(drop=True)

In [12]:
print(f"Clean articles after deduplication: {len(df)}")
print()
print("Sample of extracted fields:")
df[["title", "source", "published_date"]].head(5)

Clean articles after deduplication: 33

Sample of extracted fields:


,title,source,published_date
0,"'Humiliated, broken, powerless': Sudan enters ...",NPR,2026-04-15
1,Sexual Violence Has Become the ‘Defining Featu...,Jezebel,2026-04-02
2,Sudan war ‘being fought on women’s bodies’: Su...,Al Jazeera English,2026-03-31
3,"After three years of war, what is the situatio...",Al Jazeera English,2026-04-14
4,How I harness research to inform humanitarian ...,Nature.com,2026-04-13


In [13]:
df.tail (10)

,title,source,published_date,description,url
23,Lest We Forget the Horrors: An Unending Catalo...,Mcsweeneys.net,2026-04-22,"Early in President Trump’s first term, McSween...",https://www.mcsweeneys.net/articles/march-2026...
24,Pakistan Walks a Tightrope on Iran,Foreign Policy,2026-04-10,"As Islamabad hosts peace talks, it’s also bala...",http://foreignpolicy.com/2026/04/10/pakistan-u...
25,Middle East war: UN initiatives support mediat...,Globalsecurity.org,2026-03-30,Just hours after war broke out in the Middle E...,https://www.globalsecurity.org/wmd/library/new...
26,"India watches, Pakistan acts: The optics war D...",The Times of India,2026-04-22,Pakistan's diplomatic engagement with Iran is ...,https://economictimes.indiatimes.com/opinion/e...
27,World News in Brief: Death on the Mediterranea...,Ungeneva.org,2026-04-08,World News in Brief: Death on the Mediterranea...,https://www.ungeneva.org/en/news-media/news/20...
28,Men 18-25 to Be Automatically Registered for D...,Ibtimes.com.au,2026-04-09,WASHINGTON — Nearly all young men in the Unite...,https://www.ibtimes.com.au/men-18-25-automatic...
29,What It’s Like: Stories From Philly’s Medical ...,phillymag.com,2026-04-25,"Ever since 1751, when Benjamin Franklin and ph...",https://www.phillymag.com/be-well-philly/healt...
30,On the Brink of Global Recession,The Atlantic,2026-04-22,The economist Adam Posen on the effect of the ...,https://www.theatlantic.com/podcasts/2026/04/d...
31,"A year after his death, a mobile health clinic...",TheJournal.ie,2026-04-21,"Months before his passing, Pope Francis gave h...",https://www.thejournal.ie/mobile-health-clinic...
32,The Next Shift: The Fall of Industry and the R...,Dokumen.pub,2026-04-25,The Next Shift: The Fall of Industry and the R...,https://dokumen.pub/the-next-shift-the-fall-of...


###Clasify each of the articles
Each article is classified using keyword scoring:

1. The article's title and description are combined into a single lowercase text string
2. Each category has a list of associated keywords
3. The text is scanned for each keyword and a score is tallied per category
4. The category with the highest score is assigned to the article
5. If no keywords match, the article is classified as General

This method is chosen because it is transparent since the keyword lists are visible and editable, and it is auditable since every classification can be traced to specific matching words

In [14]:
# Five categories relevant to perational contexts.
# Keywords were selected based on published operational focus areas
# and common humanitarian news vocabulary.

CATEGORY_KEYWORDS = {

    "Health and Disease": [
        "disease", "outbreak", "epidemic", "pandemic", "infection",
        "cholera", "malaria", "tuberculosis", "tb", "hiv", "aids",
        "measles", "mpox", "monkeypox", "dengue", "typhoid",
        "vaccination", "vaccine", "immunisation", "immunization",
        "clinic", "hospital", "medical", "patient", "treatment",
        "health worker", "doctor", "nurse", "surgery", "medicine",
        "malnutrition", "nutrition", "maternal", "infant mortality",
        "mental health", "trauma", "wound", "injury",
    ],

    "Conflict and Security": [
        "war", "conflict", "armed", "attack", "violence", "fighting",
        "militia", "troops", "military", "bombing", "airstrike",
        "ceasefire", "siege", "killed", "casualties", "shelling",
        "rebel", "insurgent", "coup", "offensive", "frontline",
        "security", "threat", "evacuation", "hostage", "kidnap",
        "landmine", "explosive", "weapon",
    ],

    "Natural Disaster": [
        "flood", "flooding", "earthquake", "drought", "famine",
        "cyclone", "hurricane", "storm", "wildfire", "fire",
        "landslide", "volcano", "tsunami", "disaster",
        "climate", "weather", "monsoon", "heatwave",
        "crop failure", "food insecurity", "water shortage",
    ],

    "Humanitarian Response": [
        "humanitarian", "aid", "relief", "msf", "medecins", "ngo",
        "united nations", "unhcr", "unicef", "wfp", "who", "icrc",
        "refugee", "displaced", "idp", "asylum", "migration",
        "shelter", "food distribution", "water sanitation",
        "emergency response", "rescue", "evacuation", "donation",
        "volunteer", "camp", "settlement",
    ],

}

In [15]:
def classify_article(title, description):
    """
    Classify an article by scoring keyword matches across all categories.
    Returns the category name with the highest score, or 'General' if no match.
    """
    # Combine title and description into one searchable string
    combined_text = (str(title) + " " + str(description)).lower()

    # Count keyword matches per category
    scores = {}
    for category, keywords in CATEGORY_KEYWORDS.items():
        score = sum(1 for keyword in keywords if keyword in combined_text)
        scores[category] = score

    # Find the category with the highest score
    best_category = max(scores, key=scores.get)
    best_score    = scores[best_category]

    # If no keywords matched at all, label as General
    return best_category if best_score > 0 else "General"


In [16]:
# Apply classification to every article
df["category"] = df.apply(
    lambda row: classify_article(row["title"], row["description"]),
    axis=1
)

print("Classification done.")
print()
print("Category distribution:")
print(df["category"].value_counts().to_string())
print(f"\nTotal classified: {len(df)}")

Classification done.

Category distribution:
category
Conflict and Security    13
Humanitarian Response     7
Health and Disease        7
General                   5
Natural Disaster          1

Total classified: 33


###Preview the results now

In [18]:
# Select the columns in the order required for the output
OUTPUT_COLS = [
    "title",
    "source",
    "published_date",
    "description",
    "url",
    "category",
]


In [19]:
df_output = df[OUTPUT_COLS].copy()

In [20]:
print(f"Final dataset: {len(df_output)} articles")
print(f"Columns: {list(df_output.columns)}")
print()

# Show 5 sample rows
df_output.head(5)

Final dataset: 33 articles
Columns: ['title', 'source', 'published_date', 'description', 'url', 'category']



,title,source,published_date,description,url,category
0,"'Humiliated, broken, powerless': Sudan enters ...",NPR,2026-04-15,While parts of Sudan's capital show fragile si...,https://www.npr.org/2026/04/15/nx-s1-5781032/s...,Conflict and Security
1,Sexual Violence Has Become the ‘Defining Featu...,Jezebel,2026-04-02,"“Apart from the rapes, they beat us with stick...",https://www.jezebel.com/three-years-on-sexual-...,Conflict and Security
2,Sudan war ‘being fought on women’s bodies’: Su...,Al Jazeera English,2026-03-31,"In a new report, Doctors Without Borders says ...",https://www.aljazeera.com/news/2026/3/31/sudan...,Conflict and Security
3,"After three years of war, what is the situatio...",Al Jazeera English,2026-04-14,Sudan’s war has displaced millions and caused ...,https://www.aljazeera.com/news/2026/4/14/after...,Conflict and Security
4,How I harness research to inform humanitarian ...,Nature.com,2026-04-13,As a social scientist at Médecins Sans Frontiè...,https://www.nature.com/articles/d41586-026-001...,Humanitarian Response


###Save the files to both CSV and JSON

In [21]:
# Export as CSV
csv_filename = "Humanitarian news.csv"
df_output.to_csv(csv_filename, index=False, encoding="utf-8")
print(f"CSV saved: {csv_filename}  ({len(df_output)} rows)")

CSV saved: Humanitarian news.csv  (33 rows)


In [22]:
# Export as JSON
json_filename = "Humanitarian news.json"
df_output.to_json(json_filename, orient="records", indent=2, force_ascii=False)
print(f"JSON saved: {json_filename}  ({len(df_output)} records)")

JSON saved: Humanitarian news.json  (33 records)


In [23]:
print()
print("The completed pipeline")
print(f"Articles retrieved:    {len(all_raw_articles)}")
print(f"After deduplication:   {len(df_output)}")
print(f"Categories assigned:   {df_output['category'].nunique()}")
print()
print("Category breakdown:")
for cat, count in df_output["category"].value_counts().items():
    print(f"  {cat:<30} {count}")
print()



The completed pipeline
Articles retrieved:    35
After deduplication:   33
Categories assigned:   5

Category breakdown:
  Conflict and Security          13
  Humanitarian Response          7
  Health and Disease             7
  General                        5
  Natural Disaster               1



In [25]:
! jupyter nbconvert --to html /content/Code_assessment_2.ipynb

[NbConvertApp] Converting notebook /content/Code_assessment_2.ipynb to html
[NbConvertApp] Writing 349460 bytes to /content/Code_assessment_2.html
